# Model optimization for transaction categorization (CPU only)
This notebook is a simplified, project-specific model optimization notebook for Chameleon. It compares eager PyTorch, compiled PyTorch, ONNX Runtime CPU, and ONNX Runtime dynamic quantization on a small synthetic transaction classifier.

In [1]:
import os, time, json, random, tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import onnxruntime as ort
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import psutil

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)


## 1. Generate a synthetic transaction-like dataset

In [2]:
cats = ["groceries","restaurants","transportation","rent","shopping","travel","utilities","subscriptions"]
merchant_map = {
    "WHOLE FOODS":"groceries",
    "TRADER JOES":"groceries",
    "UBER EATS":"restaurants",
    "STARBUCKS":"restaurants",
    "UBER TRIP":"transportation",
    "LYFT":"transportation",
    "LANDLORD LLC":"rent",
    "AMAZON":"shopping",
    "TARGET":"shopping",
    "DELTA AIR":"travel",
    "MARRIOTT":"travel",
    "DUKE ENERGY":"utilities",
    "NETFLIX":"subscriptions",
}

rows = []
for i in range(3000):
    merchant = random.choice(list(merchant_map))
    y = merchant_map[merchant]
    amount = {
        "groceries": np.random.normal(85, 20),
        "restaurants": np.random.normal(18, 6),
        "transportation": np.random.normal(24, 8),
        "rent": np.random.normal(1800, 80),
        "shopping": np.random.normal(60, 25),
        "travel": np.random.normal(350, 120),
        "utilities": np.random.normal(110, 30),
        "subscriptions": np.random.normal(14, 4),
    }[y]
    dow = random.randint(0, 6)
    credit = 1 if y in {"restaurants","shopping","travel","subscriptions","transportation"} else 0
    rows.append({
        "merchant_id": list(merchant_map).index(merchant),
        "amount": float(max(1, amount)),
        "dow": dow,
        "credit": credit,
        "label": cats.index(y),
    })

df = pd.DataFrame(rows)
X = df[["merchant_id","amount","dow","credit"]].astype(np.float32).values
y = df["label"].astype(np.int64).values

scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train)
X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test)

X_train.shape, X_test.shape


((2400, 4), (600, 4))

## 2. Train a tiny surrogate PyTorch classifier

In [3]:
class TinyClassifier(nn.Module):
    def __init__(self, in_dim=4, hidden=32, out_dim=len(cats)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return self.net(x)

model = TinyClassifier()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(25):
    model.train()
    opt.zero_grad()
    logits = model(X_train_t)
    loss = loss_fn(logits, y_train_t)
    loss.backward()
    opt.step()

model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    top1 = (logits.argmax(dim=1) == y_test_t).float().mean().item()
    top3 = (logits.topk(3, dim=1).indices == y_test_t.unsqueeze(1)).any(dim=1).float().mean().item()
print({"top1_accuracy": round(top1,4), "top3_hit_rate": round(top3,4)})


{'top1_accuracy': 0.8717, 'top3_hit_rate': 1.0}


## 3. Export ONNX and create a dynamic-quantized ONNX model

In [4]:
onnx_path = "models/tx_classifier.onnx"
quant_path = "models/tx_classifier_quant_dynamic.onnx"

dummy = torch.randn(1, 4)
torch.onnx.export(
    model,
    dummy,
    onnx_path,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)

from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(onnx_path, quant_path, weight_type=QuantType.QInt8)

print("Saved:", onnx_path, quant_path)


Saved: models/tx_classifier.onnx models/tx_classifier_quant_dynamic.onnx


## 4. Benchmark helpers

In [5]:
import time
import numpy as np
import torch
import onnxruntime as ort
import psutil
import os

_process = psutil.Process(os.getpid())
_NUM_CPUS = psutil.cpu_count(logical=True) or 1

def benchmark_torch(torch_model, X_test, y_test, compiled=False, trials=200, batch_size=1):
    m = torch_model
    if compiled:
        m = torch.compile(torch_model)
        _ = m(torch.tensor(X_test[:batch_size], dtype=torch.float32))  # warm-up compile

    x_full = torch.tensor(X_test, dtype=torch.float32)
    batch = x_full[:batch_size]

    _ = m(batch)  # warm-up

    # Prime CPU percent measurement
    _process.cpu_percent(interval=None)

    times = []
    cpu_samples = []
    mem_samples = []

    for _ in range(trials):
        mem_before = _process.memory_percent()

        start = time.perf_counter()
        _ = m(batch)
        end = time.perf_counter()

        # Normalize process CPU% by logical CPU count
        cpu_now = psutil.cpu_percent(interval=0.1)
        mem_after = _process.memory_percent()

        times.append((end - start) * 1000.0)
        cpu_samples.append(cpu_now)
        mem_samples.append(max(mem_before, mem_after))

    p50 = float(np.median(times))
    p95 = float(np.percentile(times, 95))
    throughput = (batch_size * trials) / (sum(times) / 1000.0)

    with torch.no_grad():
        logits = m(x_full).cpu().numpy()
        top1 = (np.argmax(logits, axis=1) == y_test).mean()
        top3 = np.any(np.argsort(logits, axis=1)[:, -3:] == y_test[:, None], axis=1).mean()

    return {
        "top1_accuracy": round(float(top1), 4),
        "top3_hit_rate": round(float(top3), 4),
        "p50_latency_ms": round(p50, 4),
        "p95_latency_ms": round(p95, 4),
        "throughput_txns_per_sec": round(throughput, 2),
        "batch_size": batch_size,
        "cpu_pct_min": round(float(np.min(cpu_samples)), 2),
        "cpu_pct_avg": round(float(np.mean(cpu_samples)), 2),
        "cpu_pct_max": round(float(np.max(cpu_samples)), 2),
        "mem_pct_min": round(float(np.min(mem_samples)), 2),
        "mem_pct_avg": round(float(np.mean(mem_samples)), 2),
        "mem_pct_max": round(float(np.max(mem_samples)), 2),
    }


def benchmark_onnx(path, X_test, y_test, trials=200, batch_size=1):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    input_name = sess.get_inputs()[0].name

    batch = X_test[:batch_size].astype(np.float32)
    x_full = X_test.astype(np.float32)

    _ = sess.run(None, {input_name: batch})  # warm-up

    # Prime CPU percent measurement
    _process.cpu_percent(interval=None)

    times = []
    cpu_samples = []
    mem_samples = []

    for _ in range(trials):
        mem_before = _process.memory_percent()

        start = time.perf_counter()
        _ = sess.run(None, {input_name: batch})
        end = time.perf_counter()

        cpu_now = psutil.cpu_percent(interval=0.1)
        mem_after = _process.memory_percent()

        times.append((end - start) * 1000.0)
        cpu_samples.append(cpu_now)
        mem_samples.append(max(mem_before, mem_after))

    p50 = float(np.median(times))
    p95 = float(np.percentile(times, 95))
    throughput = (batch_size * trials) / (sum(times) / 1000.0)

    logits = sess.run(None, {input_name: x_full})[0]
    top1 = (np.argmax(logits, axis=1) == y_test).mean()
    top3 = np.any(np.argsort(logits, axis=1)[:, -3:] == y_test[:, None], axis=1).mean()

    return {
        "top1_accuracy": round(float(top1), 4),
        "top3_hit_rate": round(float(top3), 4),
        "p50_latency_ms": round(p50, 4),
        "p95_latency_ms": round(p95, 4),
        "throughput_txns_per_sec": round(throughput, 2),
        "batch_size": batch_size,
        "cpu_pct_min": round(float(np.min(cpu_samples)), 2),
        "cpu_pct_avg": round(float(np.mean(cpu_samples)), 2),
        "cpu_pct_max": round(float(np.max(cpu_samples)), 2),
        "mem_pct_min": round(float(np.min(mem_samples)), 2),
        "mem_pct_avg": round(float(np.mean(mem_samples)), 2),
        "mem_pct_max": round(float(np.max(mem_samples)), 2),
    }

## 5. Run the four model-level variants

In [7]:
import os

def size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)

results = []

for bs in [1, 8]:
    results.append({
        **benchmark_torch(model, X_test, y_test, compiled=False, batch_size=bs),
        "option": "torch_eager_cpu",
        "artifact_size_mb": np.nan,
    })

    results.append({
        **benchmark_torch(model, X_test, y_test, compiled=True, batch_size=bs),
        "option": "torch_compiled_cpu",
        "artifact_size_mb": np.nan,
    })

    results.append({
        **benchmark_onnx(onnx_path, X_test, y_test, batch_size=bs),
        "option": "onnx_cpu",
        "artifact_size_mb": round(size_mb(onnx_path), 4),
    })

    results.append({
        **benchmark_onnx(quant_path, X_test, y_test, batch_size=bs),
        "option": "onnx_dynamic_quant_cpu",
        "artifact_size_mb": round(size_mb(quant_path), 4),
    })

results_df = pd.DataFrame(results)

results_df = results_df[
    [
        "option",
        "batch_size",
        "top1_accuracy",
        "top3_hit_rate",
        "p50_latency_ms",
        "p95_latency_ms",
        "throughput_txns_per_sec",
        "cpu_pct_min",
        "cpu_pct_avg",
        "cpu_pct_max",
        "mem_pct_min",
        "mem_pct_avg",
        "mem_pct_max",
        "artifact_size_mb",
    ]
]

results_df

,option,batch_size,top1_accuracy,top3_hit_rate,p50_latency_ms,p95_latency_ms,throughput_txns_per_sec,cpu_pct_min,cpu_pct_avg,cpu_pct_max,mem_pct_min,mem_pct_avg,mem_pct_max,artifact_size_mb
0,torch_eager_cpu,1,0.8717,1.0,0.2498,0.3204,3722.35,0.0,0.49,15.8,18.24,18.24,18.24,NaN
1,torch_compiled_cpu,1,0.8717,1.0,0.5407,0.5900,1809.21,0.0,0.31,9.5,18.24,18.25,18.25,NaN
2,onnx_cpu,1,0.8717,1.0,0.0842,0.1364,11025.96,0.0,0.51,20.0,18.25,18.25,18.25,0.0021
3,onnx_dynamic_quant_cpu,1,0.8717,1.0,0.1133,0.1697,8199.53,0.0,0.32,15.0,18.26,18.26,18.26,0.0028
4,torch_eager_cpu,8,0.8717,1.0,0.3145,0.3815,24441.96,0.0,0.41,10.0,18.26,18.26,18.26,NaN
5,torch_compiled_cpu,8,0.8717,1.0,0.5683,0.6245,13912.64,0.0,0.54,15.0,18.27,18.27,18.27,NaN
6,onnx_cpu,8,0.8717,1.0,0.0944,0.1465,81171.32,0.0,0.29,15.0,18.27,18.27,18.27,0.0021
7,onnx_dynamic_quant_cpu,8,0.8717,1.0,0.1127,0.1674,65283.06,0.0,0.52,14.3,18.27,18.27,18.27,0.0028


## 6. Save summary for your serving-options table

In [ ]:
results_df.to_csv("results/model_optimization_summary.csv", index=False)
results_df.to_json("results/model_optimization_summary.json", orient="records", indent=2)
results_df